pytest is the de-facto standard testing framework for Python. It discovers and runs tests automatically, produces readable failure messages, and offers a fixture system that makes setup and teardown composable. This notebook starts from the basics — writing and running your first test — and builds through fixtures, parametrisation, mocking, and coverage to the patterns used in real projects.

## Installation and Discovery

Install pytest with pip:

```bash
pip install pytest
```

pytest discovers tests by:
- Recursing into directories (skipping those in `norecursedirs`)
- Collecting files matching `test_*.py` or `*_test.py`
- Collecting functions prefixed with `test_`
- Collecting methods prefixed with `test_` inside classes prefixed with `Test`

Run tests from the project root:

```bash
pytest                    # run all discovered tests
pytest tests/             # run tests in a specific directory
pytest tests/test_math.py # run a specific file
pytest -v                 # verbose — show each test name
pytest -k "addition"      # run tests whose name contains "addition"
pytest -x                 # stop after first failure
pytest --tb=short         # shorter traceback format
```

## Writing Your First Tests

A test is a plain function whose name starts with `test_`. Use Python's built-in `assert` statement — pytest rewrites it to produce detailed failure messages.

In [ ]:
# The code under test — save as src/math_utils.py in a real project
def add(a, b):
    return a + b

def divide(a, b):
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

def is_palindrome(s: str) -> bool:
    s = s.lower().replace(" ", "")
    return s == s[::-1]

def clamp(value: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, value))

In [ ]:
# test_math_utils.py — in a real project this lives in a tests/ directory

def test_add_positive_numbers():
    assert add(2, 3) == 5

def test_add_negative_numbers():
    assert add(-1, -4) == -5

def test_add_zero():
    assert add(10, 0) == 10

def test_divide_normal():
    assert divide(10, 2) == 5.0

def test_divide_float_result():
    assert divide(1, 3) == pytest.approx(0.333, rel=1e-3)

def test_is_palindrome_simple():
    assert is_palindrome("racecar")

def test_is_palindrome_with_spaces():
    assert is_palindrome("A man a plan a canal Panama")

def test_is_palindrome_false():
    assert not is_palindrome("hello")

def test_clamp_within_range():
    assert clamp(5, 0, 10) == 5

def test_clamp_below_min():
    assert clamp(-3, 0, 10) == 0

def test_clamp_above_max():
    assert clamp(15, 0, 10) == 10

# Run all tests defined above in this notebook
import pytest
pytest.main(["-v", __file__])

## pytest.approx — Floating-Point Comparisons

Floating-point arithmetic is imprecise. Never use `==` to compare floats in tests — use `pytest.approx`, which allows a small relative or absolute tolerance.

In [ ]:
import pytest

# Direct == fails due to floating-point representation
print(0.1 + 0.2 == 0.3)            # False

# pytest.approx passes
print(0.1 + 0.2 == pytest.approx(0.3))        # True  (rel tolerance 1e-6)

# Custom tolerances
print(1.001 == pytest.approx(1.0, rel=1e-2))   # True  (1% relative)
print(1.001 == pytest.approx(1.0, abs=1e-2))   # True  (absolute 0.01)

# Works with lists and dicts
results = [0.1 + 0.2, 1.0 / 3.0]
print(results == pytest.approx([0.3, 0.3333], rel=1e-3))  # True

## Testing Exceptions

Use `pytest.raises` as a context manager to assert that a specific exception is raised. You can also inspect the exception value.

In [ ]:
def test_divide_by_zero_raises():
    with pytest.raises(ValueError):
        divide(10, 0)

def test_divide_by_zero_message():
    with pytest.raises(ValueError, match="Cannot divide by zero"):
        divide(10, 0)

def test_divide_by_zero_exc_info():
    with pytest.raises(ValueError) as exc_info:
        divide(5, 0)
    assert "zero" in str(exc_info.value).lower()

# Verify the exception type specifically
def test_type_error_on_bad_input():
    with pytest.raises(TypeError):
        add("a", 1)   # str + int raises TypeError

# Run the tests
test_divide_by_zero_raises()
test_divide_by_zero_message()
test_divide_by_zero_exc_info()
print("All exception tests passed")

## Fixtures

A **fixture** is a function decorated with `@pytest.fixture`. It provides a test with a resource — a database connection, a populated object, a temp file — and handles setup and teardown automatically.

Tests declare what fixtures they need by naming them as parameters. pytest injects them.

In [ ]:
# A simple in-memory "database" for demonstration
class UserStore:
    def __init__(self):
        self._users = {}
        self._next_id = 1

    def add(self, name: str, email: str) -> dict:
        user = {"id": self._next_id, "name": name, "email": email}
        self._users[self._next_id] = user
        self._next_id += 1
        return user

    def get(self, user_id: int) -> dict | None:
        return self._users.get(user_id)

    def delete(self, user_id: int) -> bool:
        return self._users.pop(user_id, None) is not None

    def count(self) -> int:
        return len(self._users)

    def all(self) -> list[dict]:
        return list(self._users.values())


@pytest.fixture
def empty_store():
    """A fresh empty UserStore."""
    return UserStore()


@pytest.fixture
def populated_store():
    """A UserStore with three users already added."""
    store = UserStore()
    store.add("Alice", "alice@example.com")
    store.add("Bob",   "bob@example.com")
    store.add("Carol", "carol@example.com")
    return store


def test_empty_store_has_zero_users(empty_store):
    assert empty_store.count() == 0


def test_add_user(empty_store):
    user = empty_store.add("Dave", "dave@example.com")
    assert user["id"] == 1
    assert user["name"] == "Dave"
    assert empty_store.count() == 1


def test_populated_store_count(populated_store):
    assert populated_store.count() == 3


def test_get_existing_user(populated_store):
    user = populated_store.get(1)
    assert user["name"] == "Alice"


def test_delete_user(populated_store):
    assert populated_store.delete(2) is True
    assert populated_store.count() == 2
    assert populated_store.get(2) is None


# Run
store_e = empty_store()
store_p = populated_store()
test_empty_store_has_zero_users(store_e)
test_add_user(UserStore())
test_populated_store_count(store_p)
test_get_existing_user(store_p)
test_delete_user(populated_store())
print("All fixture tests passed")

## Fixture Teardown with `yield`

Use `yield` in a fixture to separate setup (before `yield`) from teardown (after `yield`). Code after `yield` runs regardless of whether the test passes or fails — like a `finally` block.

In [ ]:
import tempfile
import os

@pytest.fixture
def temp_file():
    """Create a temporary file; delete it after the test."""
    fd, path = tempfile.mkstemp(suffix=".txt")
    os.close(fd)
    print(f"\n[setup] created {path}")

    yield path                          # test receives the file path

    print(f"[teardown] removing {path}")
    os.unlink(path)                     # always runs, even if test fails


def test_write_and_read(temp_file):
    # Write data
    with open(temp_file, "w") as f:
        f.write("hello pytest")

    # Read it back
    with open(temp_file) as f:
        content = f.read()

    assert content == "hello pytest"


# Demonstrate manually
path = None
for path in [next(temp_file.__wrapped__() if hasattr(temp_file, '__wrapped__') else iter([temp_file()]))]: break

# Simpler demonstration
tf_gen = (lambda: (_ for _ in ()))()
tmpfd, tmppath = tempfile.mkstemp(suffix=".txt")
os.close(tmpfd)
try:
    with open(tmppath, "w") as f:
        f.write("hello pytest")
    with open(tmppath) as f:
        assert f.read() == "hello pytest"
    print("temp_file fixture pattern works")
finally:
    os.unlink(tmppath)

## Fixture Scope

By default, a fixture is recreated for every test. The `scope` parameter controls how long a fixture lives:

| Scope | Created once per… |
|---|---|
| `function` (default) | Each test function |
| `class` | Each test class |
| `module` | Each test file |
| `session` | Entire test run |

Use wider scopes for expensive setup (database connections, server processes) that can safely be shared.

In [ ]:
@pytest.fixture(scope="module")
def db_connection():
    """Simulate an expensive DB connection — created once per module."""
    print("\n[module setup] opening DB connection")
    conn = {"status": "open", "queries": 0}
    yield conn
    print("\n[module teardown] closing DB connection")
    conn["status"] = "closed"


@pytest.fixture
def fresh_cursor(db_connection):
    """Function-scoped fixture that uses the module-scoped connection."""
    db_connection["queries"] += 1
    return {"conn": db_connection, "cursor_id": db_connection["queries"]}


def test_first_query(fresh_cursor):
    assert fresh_cursor["conn"]["status"] == "open"
    assert fresh_cursor["cursor_id"] == 1


def test_second_query(fresh_cursor):
    # db_connection is reused; cursor_id increments
    assert fresh_cursor["cursor_id"] == 2


# Demonstrate fixture scope behaviour
conn = {"status": "open", "queries": 0}
cursor1 = {"conn": conn, "cursor_id": (conn.__setitem__("queries", 1) or 1)}
cursor2 = {"conn": conn, "cursor_id": (conn.__setitem__("queries", 2) or 2)}
print(f"cursor1 id: {cursor1['cursor_id']}, cursor2 id: {cursor2['cursor_id']}")
print(f"Shared connection queries: {conn['queries']}")

## Parametrize — Running One Test with Many Inputs

`@pytest.mark.parametrize` runs a test function multiple times with different arguments. Each combination appears as a separate test in the output — and a failure in one does not prevent the others from running.

In [ ]:
@pytest.mark.parametrize("a, b, expected", [
    (1, 2, 3),
    (0, 0, 0),
    (-1, 1, 0),
    (100, -50, 50),
    (0.1, 0.2, pytest.approx(0.3)),
])
def test_add_parametrized(a, b, expected):
    assert add(a, b) == expected


@pytest.mark.parametrize("text, expected", [
    ("racecar",  True),
    ("hello",    False),
    ("A man a plan a canal Panama", True),
    ("Was it a car or a cat I saw", True),
    ("Python",   False),
])
def test_palindrome_parametrized(text, expected):
    assert is_palindrome(text) == expected


# Run the parametrized tests manually
for a, b, expected in [(1,2,3),(0,0,0),(-1,1,0),(100,-50,50)]:
    result = add(a, b)
    assert result == expected, f"add({a},{b}) = {result}, expected {expected}"
    print(f"add({a:4}, {b:4}) = {result:4}  ✓")

for text, expected in [("racecar",True),("hello",False),("A man a plan a canal Panama",True)]:
    assert is_palindrome(text) == expected
    print(f"palindrome({text!r:35}) = {expected}  ✓")

In [ ]:
# Parametrize with IDs — custom names for each test case
@pytest.mark.parametrize("value, lo, hi, expected", [
    (5,   0, 10, 5),
    (-3,  0, 10, 0),
    (15,  0, 10, 10),
    (0,   0, 10, 0),
    (10,  0, 10, 10),
], ids=["in-range", "below-min", "above-max", "at-min", "at-max"])
def test_clamp_parametrized(value, lo, hi, expected):
    assert clamp(value, lo, hi) == expected


# Multiple parametrize decorators — creates the Cartesian product
@pytest.mark.parametrize("base",   [2, 10])
@pytest.mark.parametrize("exp",    [0,  1, 3])
def test_power_parametrized(base, exp):
    assert base ** exp == pow(base, exp)   # 6 test cases: 2×3


for value, lo, hi, expected in [(5,0,10,5),(-3,0,10,0),(15,0,10,10)]:
    assert clamp(value, lo, hi) == expected
print("Clamp parametrized tests passed")

## Marks — Skipping, Expected Failures, Custom Groups

pytest marks let you attach metadata to tests. Built-in marks include `skip`, `skipif`, `xfail`, and `filterwarnings`. You can also define custom marks to group tests by category.

In [ ]:
import sys

# skip — always skip this test
@pytest.mark.skip(reason="feature not yet implemented")
def test_future_feature():
    assert False   # never runs


# skipif — conditionally skip
@pytest.mark.skipif(sys.platform == "win32", reason="POSIX-only API")
def test_posix_signal():
    import signal
    assert hasattr(signal, "SIGTERM")


# xfail — expected failure: test is allowed to fail
@pytest.mark.xfail(reason="known bug in upstream library")
def test_known_bug():
    assert 1 == 2   # fails, but marked expected — reported as 'xfail' not 'FAILED'


# xfail strict=True — test MUST fail; if it passes unexpectedly it becomes an error
@pytest.mark.xfail(strict=True, reason="bug #123 not yet fixed")
def test_strict_xfail():
    assert 1 == 2   # correctly fails


# Custom marks — group tests by category (register in pytest.ini / pyproject.toml)
@pytest.mark.slow
def test_slow_integration():
    import time
    time.sleep(0.01)
    assert True


@pytest.mark.unit
def test_fast_unit():
    assert add(1, 1) == 2


# Run only 'unit' tests: pytest -m unit
# Skip 'slow' tests:    pytest -m "not slow"
print("Mark examples defined (run with pytest to see skip/xfail behaviour)")

## Built-in Fixtures: `tmp_path`, `capsys`, `monkeypatch`

pytest ships with many useful built-in fixtures. The three most commonly used are:

- `tmp_path` — provides a `pathlib.Path` to a temporary directory unique to each test
- `capsys` — captures stdout and stderr output
- `monkeypatch` — temporarily replaces attributes, environment variables, or functions

In [ ]:
from pathlib import Path

# tmp_path — no cleanup code needed; pytest handles it
def test_write_config(tmp_path):
    config = tmp_path / "config.ini"
    config.write_text("[settings]\ndebug=true\n")

    content = config.read_text()
    assert "debug=true" in content
    assert config.exists()


def test_directory_structure(tmp_path):
    (tmp_path / "src").mkdir()
    (tmp_path / "src" / "main.py").write_text("print('hello')")
    (tmp_path / "tests").mkdir()

    assert (tmp_path / "src" / "main.py").exists()
    assert len(list(tmp_path.iterdir())) == 2  # src and tests dirs


# Demonstrate tmp_path pattern
tmpdir = Path(tempfile.mkdtemp())
try:
    cfg = tmpdir / "config.ini"
    cfg.write_text("[settings]\ndebug=true\n")
    assert "debug=true" in cfg.read_text()
    print("tmp_path pattern works")
finally:
    import shutil
    shutil.rmtree(tmpdir)

In [ ]:
# capsys — capture print output
def greet_stdout(name):
    print(f"Hello, {name}!")
    print(f"Welcome.", end="")


def test_greet_stdout(capsys):
    greet_stdout("Alice")
    captured = capsys.readouterr()     # returns (out, err) namedtuple
    assert captured.out == "Hello, Alice!\nWelcome."
    assert captured.err == ""


# Demonstrate capsys pattern using io.StringIO
import io, contextlib

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    greet_stdout("Alice")
output = buf.getvalue()
assert output == "Hello, Alice!\nWelcome."
print(f"Captured: {output!r}")

In [ ]:
# monkeypatch — temporarily replace values; restored after the test
import os

def get_database_url():
    return os.environ.get("DATABASE_URL", "sqlite:///default.db")


def test_database_url_from_env(monkeypatch):
    monkeypatch.setenv("DATABASE_URL", "postgres://localhost/testdb")
    assert get_database_url() == "postgres://localhost/testdb"
    # env var is restored to its original value after the test


def test_database_url_default(monkeypatch):
    monkeypatch.delenv("DATABASE_URL", raising=False)
    assert get_database_url() == "sqlite:///default.db"


# monkeypatch.setattr — replace an attribute or function
import time as time_module

def get_timestamp():
    return time_module.time()


def test_mocked_time(monkeypatch):
    monkeypatch.setattr(time_module, "time", lambda: 1_700_000_000.0)
    assert get_timestamp() == 1_700_000_000.0


# Demonstrate monkeypatch pattern manually
original = os.environ.get("DATABASE_URL")
os.environ["DATABASE_URL"] = "postgres://localhost/testdb"
try:
    assert get_database_url() == "postgres://localhost/testdb"
    print("monkeypatch pattern works")
finally:
    if original is None:
        os.environ.pop("DATABASE_URL", None)
    else:
        os.environ["DATABASE_URL"] = original

## Mocking with `unittest.mock`

`unittest.mock.Mock` and `MagicMock` create objects that record all calls made to them. `patch` temporarily replaces a name in a module — essential for isolating code from external services.

In [ ]:
from unittest.mock import Mock, MagicMock, patch, call

# Basic Mock — records calls, returns Mock by default
m = Mock()
m(1, 2, key="value")
m.some_method()

print(m.call_count)                    # 1
print(m.call_args)                     # call(1, 2, key='value')
m.assert_called_once_with(1, 2, key="value")
m.some_method.assert_called_once()

# Configure return values
m.return_value = 42
print(m("anything"))                   # 42

m.side_effect = [10, 20, 30]
print(m(), m(), m())                   # 10 20 30

# side_effect can also be an exception
m.side_effect = ValueError("bad input")
try:
    m()
except ValueError as e:
    print(f"Got expected error: {e}")

In [ ]:
# patch — replace a name in a module during the test
# The code under test:
def send_email(to: str, subject: str, body: str) -> bool:
    """In production, calls an external email service."""
    import smtplib   # expensive import, external service
    # ... real implementation would connect and send
    return True


def notify_user(user_email: str, message: str) -> dict:
    """Business logic that depends on send_email."""
    success = send_email(
        to=user_email,
        subject="Notification",
        body=message,
    )
    return {"sent": success, "to": user_email}


# Test without touching the real email service
with patch("__main__.send_email") as mock_send:
    mock_send.return_value = True

    result = notify_user("alice@example.com", "Your order shipped")

    assert result == {"sent": True, "to": "alice@example.com"}
    mock_send.assert_called_once_with(
        to="alice@example.com",
        subject="Notification",
        body="Your order shipped",
    )
    print(f"send_email called: {mock_send.call_count} time(s)")
    print(f"args: {mock_send.call_args}")

In [ ]:
# patch as a decorator
@patch("__main__.send_email")
def test_notify_failure(mock_send):
    mock_send.return_value = False
    result = notify_user("bob@example.com", "Payment failed")
    assert result["sent"] is False
    assert result["to"] == "bob@example.com"

test_notify_failure()
print("Decorator patch test passed")


# MagicMock — like Mock but supports magic methods (__len__, __iter__, etc.)
mm = MagicMock()
mm.__len__.return_value = 5
mm.__iter__.return_value = iter([1, 2, 3])

print(len(mm))            # 5
print(list(mm))           # [1, 2, 3]

# Spec — mock that rejects attributes the real object doesn't have
from unittest.mock import create_autospec

real_store = UserStore()
mock_store = create_autospec(real_store)
mock_store.count.return_value = 7

print(mock_store.count())  # 7
try:
    mock_store.nonexistent_method()  # AttributeError — not on UserStore
except AttributeError as e:
    print(f"Spec caught invalid access: {e}")

## Testing Classes

Group related tests in a class prefixed with `Test`. The class gets no `__init__` — pytest instantiates it for each test. Fixtures still work as method parameters. A class-level fixture (via `autouse`) runs for every method in the class.

In [ ]:
class ShoppingCart:
    def __init__(self):
        self._items: dict[str, dict] = {}

    def add(self, name: str, price: float, qty: int = 1) -> None:
        if name in self._items:
            self._items[name]["qty"] += qty
        else:
            self._items[name] = {"price": price, "qty": qty}

    def remove(self, name: str) -> None:
        self._items.pop(name, None)

    def total(self) -> float:
        return sum(v["price"] * v["qty"] for v in self._items.values())

    def item_count(self) -> int:
        return sum(v["qty"] for v in self._items.values())

    def is_empty(self) -> bool:
        return len(self._items) == 0


class TestShoppingCart:
    @pytest.fixture(autouse=True)
    def cart(self):
        """Fresh cart for every test method in this class."""
        self.cart = ShoppingCart()

    def test_new_cart_is_empty(self):
        assert self.cart.is_empty()
        assert self.cart.total() == 0.0

    def test_add_single_item(self):
        self.cart.add("apple", 0.50)
        assert self.cart.item_count() == 1
        assert self.cart.total() == pytest.approx(0.50)

    def test_add_multiple_quantities(self):
        self.cart.add("apple", 0.50, qty=3)
        assert self.cart.item_count() == 3
        assert self.cart.total() == pytest.approx(1.50)

    def test_add_same_item_twice_accumulates(self):
        self.cart.add("apple", 0.50)
        self.cart.add("apple", 0.50)
        assert self.cart.item_count() == 2

    def test_remove_item(self):
        self.cart.add("banana", 0.30)
        self.cart.add("cherry", 1.20)
        self.cart.remove("banana")
        assert self.cart.item_count() == 1
        assert self.cart.total() == pytest.approx(1.20)

    def test_total_multiple_items(self):
        self.cart.add("apple",  0.50, qty=2)
        self.cart.add("banana", 0.30, qty=3)
        assert self.cart.total() == pytest.approx(1.90)


# Run class tests manually
cart = ShoppingCart()
assert cart.is_empty()
cart.add("apple", 0.50, qty=2)
cart.add("banana", 0.30, qty=3)
assert abs(cart.total() - 1.90) < 1e-9
print(f"Cart total: {cart.total():.2f}")
print("Class tests passed")

## `conftest.py` — Shared Fixtures

pytest automatically loads `conftest.py` files. Fixtures defined there are available to all tests in the same directory and subdirectories — without any import.

A typical project layout:

```
project/
├── src/
│   └── myapp/
├── tests/
│   ├── conftest.py          ← shared fixtures for all tests
│   ├── unit/
│   │   ├── conftest.py      ← fixtures for unit tests only
│   │   └── test_math.py
│   └── integration/
│       ├── conftest.py      ← fixtures for integration tests only
│       └── test_api.py
└── pyproject.toml
```

```python
# tests/conftest.py
import pytest
from myapp.db import Database

@pytest.fixture(scope="session")
def db():
    database = Database(url="sqlite:///:memory:")
    database.create_tables()
    yield database
    database.drop_tables()

@pytest.fixture
def client(db):
    from myapp.app import create_app
    app = create_app(db=db, testing=True)
    with app.test_client() as c:
        yield c
```

Tests in any subdirectory can use `db` and `client` without importing them.

## Parametrized Fixtures

A fixture can itself be parametrized. Every test that uses it runs once per parameter — multiplied by any test-level parametrize decorators.

In [ ]:
# Fixture that provides different storage backends
@pytest.fixture(params=["memory", "dict"])
def store_backend(request):
    """Tests using this fixture run once per backend."""
    if request.param == "memory":
        return UserStore()
    else:
        # Could return a different implementation — both satisfy the same interface
        store = UserStore()
        store._backend = "dict"
        return store


def test_store_add_with_backend(store_backend):
    user = store_backend.add("Alice", "alice@example.com")
    assert store_backend.count() == 1
    assert user["name"] == "Alice"


def test_store_get_with_backend(store_backend):
    store_backend.add("Bob", "bob@example.com")
    user = store_backend.get(1)
    assert user is not None


# Demonstrate — both backends get tested
for backend in ["memory", "dict"]:
    s = UserStore()
    s.add("Alice", "alice@example.com")
    assert s.count() == 1
    print(f"Backend '{backend}': ✓")

## Coverage — Measuring What Is Tested

`pytest-cov` integrates coverage measurement into pytest:

```bash
pip install pytest-cov

pytest --cov=src --cov-report=term-missing
pytest --cov=src --cov-report=html      # generates htmlcov/index.html
pytest --cov=src --cov-fail-under=80    # fail if coverage < 80%
```

A `pyproject.toml` configuration:

```toml
[tool.pytest.ini_options]
addopts = "--cov=src --cov-report=term-missing --cov-fail-under=80"

[tool.coverage.run]
branch = true
source = ["src"]

[tool.coverage.report]
exclude_lines = [
    "pragma: no cover",
    "if TYPE_CHECKING:",
    "raise NotImplementedError",
]
```

**100% coverage does not mean bug-free.** Coverage tells you which lines were executed — not whether the behaviour is correct. Aim for high coverage on business logic; accept lower coverage for glue code, CLI entrypoints, and error paths that are hard to trigger.

## `pytest.ini` / `pyproject.toml` Configuration

Configure pytest in `pyproject.toml` to avoid repeating command-line flags:

```toml
[tool.pytest.ini_options]
testpaths     = ["tests"]
addopts       = "-v --tb=short"
markers       = [
    "slow: marks tests as slow (deselect with '-m not slow')",
    "unit: fast unit tests",
    "integration: tests that hit real services",
]
filterwarnings = [
    "error",
    "ignore::DeprecationWarning:third_party_lib",
]
```

Registering custom marks avoids the `PytestUnknownMarkWarning` warning.

## Testing Patterns and Best Practices

**Arrange — Act — Assert (AAA):** Structure every test in three clear sections: set up state, perform the action, verify the result. One blank line between each section helps readability.

**One assertion per concept:** A test can have multiple assertions if they all verify the same behaviour. If a test fails, it should be immediately obvious what went wrong — not buried in assertion 7 of 12.

**Name tests like sentences:** `test_add_returns_sum_of_two_integers` is more useful than `test_add`. When a test fails in CI, the name is the first thing you read.

**Avoid logic in tests:** No `if`, `for`, or `try` in test bodies. If you need to test many cases, use parametrize.

**Keep tests independent:** Tests must not depend on each other's side effects. Each test must be able to run in isolation and in any order.

In [ ]:
# Good test structure — AAA pattern
def test_cart_total_after_discount():
    # Arrange
    cart = ShoppingCart()
    cart.add("laptop",  999.00)
    cart.add("mouse",    29.99)
    discount = 0.10

    # Act
    total_before = cart.total()
    discounted   = total_before * (1 - discount)

    # Assert
    assert total_before == pytest.approx(1028.99)
    assert discounted   == pytest.approx(926.091)


# Descriptive failure messages with assert
def test_user_has_required_fields():
    user = {"id": 1, "name": "Alice", "email": "alice@example.com"}
    required = {"id", "name", "email"}
    missing = required - user.keys()
    assert not missing, f"User dict missing fields: {missing}"


# Avoid testing implementation details — test behaviour, not internals
def test_add_user_increases_count():
    store = UserStore()
    initial = store.count()
    store.add("New User", "new@example.com")
    assert store.count() == initial + 1   # behaviour: count went up by 1
    # NOT: assert len(store._users) == 1  — that's an internal detail


test_cart_total_after_discount()
test_user_has_required_fields()
test_add_user_increases_count()
print("Best practice tests passed")

## Summary

- pytest discovers test files matching `test_*.py` and functions prefixed with `test_`. Run with `pytest -v` for verbose output, `-k` to filter by name, `-x` to stop on first failure.
- Use plain `assert` statements — pytest rewrites them to show the actual values on failure. Use `pytest.approx` for floating-point comparisons.
- `pytest.raises` asserts that a block raises a specific exception. Pass `match=` to check the message.
- **Fixtures** are functions decorated with `@pytest.fixture`. Tests declare their dependencies as parameters and pytest injects them. Use `yield` to separate setup from teardown.
- **Fixture scope** (`function`, `class`, `module`, `session`) controls how often a fixture is created. Use wider scopes for expensive resources.
- **`@pytest.mark.parametrize`** runs a test with multiple sets of arguments. Each set is a separate test case. Use `ids=` for readable names.
- Built-in marks: `skip`, `skipif`, `xfail`. Custom marks group tests; register them in `pyproject.toml` to avoid warnings.
- Built-in fixtures: `tmp_path` (temporary directory), `capsys` (capture stdout/stderr), `monkeypatch` (temporarily replace attributes, env vars, functions).
- `unittest.mock.Mock` records calls and returns configured values. `patch` replaces a name in a module for the duration of a test. `create_autospec` creates a mock that rejects attributes the real object does not have.
- Group related tests in a `Test`-prefixed class. Use `autouse=True` on a class-scoped fixture to run setup for every method.
- `conftest.py` provides shared fixtures to all tests in a directory without imports.
- `pytest-cov` measures which lines are executed. High coverage on business logic is valuable; 100% coverage does not guarantee correctness.
- Follow Arrange — Act — Assert, keep tests independent, name them like sentences, and test behaviour not implementation details.